# Punto 2 — Split Temprano y Preprocesamiento sin Data Leakage

**Dataset:** PetFinder Malaysia (Kaggle)  
**Principio:** el split train/validación se realiza **antes** de cualquier operación que aprenda estadísticas del dataset. Todo fit se hace **únicamente sobre train** y luego se aplica a validación.

```
Orden correcto:
  1. Feature engineering estructural (sin estadísticas → sin leakage)
  2. Split estratificado
  3. fit(X_train)      ← aprende mediana, frecuencias, etc.
  4. transform(X_train) y transform(X_val)
```
---

## 0. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from joblib import dump
import os
import warnings
warnings.filterwarnings('ignore')

print('Librerías importadas.')

In [ ]:
SEED       = 42
VAL_SIZE   = 0.20
BASE_PATH  = '../input/petfinder-adoption-prediction'
OUTPUT_DIR = 'output'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}/')

---
## 1. Carga de datos

In [ ]:
df_raw = pd.read_csv(f'{BASE_PATH}/train/train.csv')
print(f'Shape: {df_raw.shape}')

In [ ]:
nulos = df_raw.isnull().sum()
print('Columnas con nulos:')
print(nulos[nulos > 0])

---
## 2. Feature engineering PRE-SPLIT

Estas operaciones son **deterministas por fila**: no usan estadísticas del dataset (media, frecuencia, etc.), por lo que se pueden aplicar sobre el dataset completo **sin introducir leakage**.

In [ ]:
df = df_raw.copy()

In [ ]:
# Binaria: tiene nombre real (no nulo ni placeholder)
df['Tiene_nombre'] = df['Name'].apply(
    lambda x: 0 if (pd.isna(x) or str(x).strip().lower() in ['no name', 'no name yet']) else 1
)
print(f"Tiene_nombre — con nombre: {df['Tiene_nombre'].sum()} | sin nombre: {(df['Tiene_nombre']==0).sum()}")

In [ ]:
# Longitud de descripción en palabras
df['Desc_len_words'] = df['Description'].fillna('').apply(lambda x: len(str(x).split()))
print(f"Desc_len_words — media: {df['Desc_len_words'].mean():.1f} palabras")

In [ ]:
# Binarias simples derivadas directamente de columnas existentes
df['tiene_video'] = (df['VideoAmt'] > 0).astype(int)
df['fee_binario'] = (df['Fee'] > 0).astype(int)
df['raza_mixta']  = (df['Breed2'] > 0).astype(int)

print(f"tiene_video: {df['tiene_video'].sum()} mascotas con video")
print(f"fee_binario: {df['fee_binario'].sum()} adopciones con costo")
print(f"raza_mixta:  {df['raza_mixta'].sum()} mascotas con raza mixta")

In [ ]:
# Score de salud: suma de 4 indicadores positivos por fila (0 a 4)
df['health_score'] = (
    (df['Vaccinated'] == 1).astype(int) +
    (df['Dewormed']   == 1).astype(int) +
    (df['Sterilized'] == 1).astype(int) +
    (df['Health']     == 1).astype(int)
)
print(f"health_score — distribución:\n{df['health_score'].value_counts().sort_index()}")

---
## 3. Definición de features y target

In [ ]:
TARGET = 'AdoptionSpeed'

# Numéricas — Age tiene nulos (~8%), el resto no
NUMERIC_FEATS = [
    'Age', 'Fee', 'PhotoAmt', 'Quantity', 'VideoAmt',
    'Desc_len_words', 'Tiene_nombre', 'tiene_video',
    'fee_binario', 'raza_mixta', 'health_score'
]
print(f'Numéricas: {NUMERIC_FEATS}')

In [ ]:
# Categóricas ordinales — ya numéricas, sin nulos
CAT_ORDINAL_FEATS = [
    'Type', 'Gender', 'MaturitySize', 'FurLength',
    'Vaccinated', 'Dewormed', 'Sterilized', 'Health',
    'Color1', 'Color2', 'Color3', 'State'
]

# Alta cardinalidad — requieren encoding aprendido de train
CAT_HIGH_CARD_FEATS = ['Breed1', 'Breed2']

print(f'Ordinales: {CAT_ORDINAL_FEATS}')
print(f'Alta cardinalidad: {CAT_HIGH_CARD_FEATS}')

In [ ]:
ALL_FEATS = NUMERIC_FEATS + CAT_ORDINAL_FEATS + CAT_HIGH_CARD_FEATS

X = df[ALL_FEATS].copy()
y = df[TARGET].copy()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nNulos en X:')
nulos_x = X.isnull().sum()
print(nulos_x[nulos_x > 0])

---
## 4. Split estratificado train / validación

`stratify=y` garantiza que la proporción de cada clase de `AdoptionSpeed` sea igual en train y validación.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y
)

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}')

In [ ]:
# Verificar que las proporciones del target se mantienen
dist = pd.concat([
    y.value_counts(normalize=True).sort_index().rename('original'),
    y_train.value_counts(normalize=True).sort_index().rename('train'),
    y_val.value_counts(normalize=True).sort_index().rename('val'),
], axis=1).round(4)

print('Distribución de AdoptionSpeed por split:')
print(dist)

---
## 5. Preprocesamiento — fit SOLO en train

A partir de aquí, **ningún `fit` toca `X_val`**.

In [ ]:
X_train = X_train.copy()
X_val   = X_val.copy()

### 5.1 Outliers en `Age`

Valores > 180 meses son biológicamente improbables (del EDA). El umbral es fijo → no usa estadísticas del dataset → sin leakage.

In [ ]:
AGE_MAX = 180

X_train.loc[X_train['Age'] > AGE_MAX, 'Age'] = np.nan
X_val.loc[X_val['Age'] > AGE_MAX, 'Age']     = np.nan

print(f'Nulos en Age tras eliminar outliers — train: {X_train["Age"].isnull().sum()} | val: {X_val["Age"].isnull().sum()}')

### 5.2 Imputación de `Age` — mediana aprendida de train

In [ ]:
age_imputer = SimpleImputer(strategy='median')

# fit SOLO en train
age_imputer.fit(X_train[['Age']])
mediana_aprendida = age_imputer.statistics_[0]

print(f'Mediana aprendida de train: {mediana_aprendida:.1f} meses')

In [ ]:
# transform en ambos con la mediana de train
X_train['Age'] = age_imputer.transform(X_train[['Age']]).ravel()
X_val['Age']   = age_imputer.transform(X_val[['Age']]).ravel()

print(f'NaN en Age tras imputar — train: {X_train["Age"].isnull().sum()} | val: {X_val["Age"].isnull().sum()}')

### 5.3 Frequency encoding para `Breed1` y `Breed2`

Alta cardinalidad (176 y 135 valores únicos). Se reemplazan por la frecuencia relativa de cada valor **en train**. Valores no vistos en train → frecuencia 0.

In [ ]:
# Aprender frecuencias SOLO de train
freq_maps = {}
for col in CAT_HIGH_CARD_FEATS:
    freq_maps[col] = X_train[col].value_counts(normalize=True).to_dict()
    print(f'{col}: {len(freq_maps[col])} valores únicos en train')

In [ ]:
# Aplicar a train y val
for col in CAT_HIGH_CARD_FEATS:
    new_col = f'{col}_freq'
    X_train[new_col] = X_train[col].map(freq_maps[col]).fillna(0.0)
    X_val[new_col]   = X_val[col].map(freq_maps[col]).fillna(0.0)

    unseen = X_val[col].map(freq_maps[col]).isnull().sum()
    print(f'{col}: {unseen} valores no vistos en val (→ asignados freq=0)')

In [ ]:
# Eliminar columnas originales (reemplazadas por _freq)
X_train = X_train.drop(columns=CAT_HIGH_CARD_FEATS)
X_val   = X_val.drop(columns=CAT_HIGH_CARD_FEATS)

print(f'Shape final — train: {X_train.shape} | val: {X_val.shape}')

---
## 6. Verificación anti-leakage

In [ ]:
# Sin NaN
print(f'NaN totales — train: {X_train.isnull().sum().sum()} | val: {X_val.isnull().sum().sum()}')

In [ ]:
# La mediana usada es la de train, no la del dataset completo
print(f'Mediana aprendida de train:      {mediana_aprendida:.2f}')
print(f'Mediana real de train:           {X_train["Age"].median():.2f}')
print(f'Mediana de val (no se usó):      {X_val["Age"].median():.2f}')

In [ ]:
# Columnas finales
print(f'Columnas finales ({len(X_train.columns)}):')
print(list(X_train.columns))

---
## 7. Guardado de artefactos

In [ ]:
# Datasets procesados con target incluido
train_out = X_train.copy()
train_out[TARGET] = y_train.values

val_out = X_val.copy()
val_out[TARGET] = y_val.values

train_out.to_csv(f'{OUTPUT_DIR}/train_processed.csv', index=False)
val_out.to_csv(f'{OUTPUT_DIR}/val_processed.csv', index=False)

print(f'train_processed.csv → {train_out.shape}')
print(f'val_processed.csv   → {val_out.shape}')

In [ ]:
# Transformadores entrenados (para aplicar al test set de Kaggle)
dump(age_imputer, f'{OUTPUT_DIR}/age_imputer.joblib')
dump(freq_maps,   f'{OUTPUT_DIR}/freq_maps.joblib')

print(f'age_imputer.joblib → mediana={mediana_aprendida}')
print(f'freq_maps.joblib   → {list(freq_maps.keys())}')

---
## Resumen

| Paso | Operación | Momento | Leakage |
|---|---|:---:|:---:|
| Features estructurales | `Tiene_nombre`, `Desc_len_words`, etc. | Pre-split | No |
| Outliers Age (umbral fijo) | `Age > 180 → NaN` | Post-split | No |
| **Split** | `train_test_split(stratify=y)` | — | — |
| Imputación Age | `SimpleImputer.fit(X_train)` | Post-split | No (fit solo en train) |
| Freq encoding Breed | `value_counts` de `X_train` | Post-split | No (fit solo en train) |